# Feature Engineering: Trending Products

Build the weighted trending score for contextual product recommendations.

**Features:**
- `time_bucket` (Morning / Afternoon / Night)
- `purchase_count` per (product x day x time_bucket)
- `normalized_purchase_frequency` (min-max scaled within context)
- `reorder_rate` (proportion of purchases that are reorders)
- `contextual_frequency` (lift vs global average)
- `weighted_trending_score`

**Formula:**
```
weighted_score = 0.5 * normalized_purchase_frequency
               + 0.3 * reorder_rate
               + 0.2 * contextual_frequency
```

**Outputs:** Parquet, CSV, and Pickle files for API serving.

In [1]:
import os
import time
import pickle
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../data/combined_instacart_data.csv'
OUTPUTS_DIR = '../outputs'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

## 1. Load & Prepare Data

In [2]:
t0 = time.time()

dtype_map = {
    'order_id': 'int32', 'product_id': 'int32',
    'reordered': 'int8', 'order_dow': 'int8',
    'order_hour_of_day': 'int8',
}

use_cols = ['order_id', 'product_id', 'reordered', 'order_dow',
            'order_hour_of_day', 'product_name', 'aisle', 'department']

df = pd.read_csv(DATA_PATH, usecols=use_cols,
                 dtype={k: v for k, v in dtype_map.items() if k in use_cols})

print(f'Loaded {len(df):,} rows in {time.time()-t0:.1f}s')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
df.head(3)

Loaded 33,819,106 rows in 79.4s
Memory: 7.76 GB


,order_id,product_id,reordered,order_dow,order_hour_of_day,product_name,aisle,department
0,2,33120,1,5,9,Organic Egg Whites,eggs,dairy eggs
1,2,28985,1,5,9,Michigan Organic Kale,fresh vegetables,produce
2,2,9327,0,5,9,Garlic Powder,spices seasonings,pantry


## 2. Create Time Bucket Feature

In [3]:
# Time bucket mapping
TIME_BUCKET_MAP = {}
for h in range(24):
    if 5 <= h <= 11:
        TIME_BUCKET_MAP[h] = 'morning'
    elif 12 <= h <= 17:
        TIME_BUCKET_MAP[h] = 'afternoon'
    else:
        TIME_BUCKET_MAP[h] = 'night'

df['time_bucket'] = df['order_hour_of_day'].map(TIME_BUCKET_MAP)

print('Time bucket distribution:')
print(df['time_bucket'].value_counts().to_frame('count').assign(
    pct=lambda x: (x['count'] / x['count'].sum() * 100).round(1)
))

Time bucket distribution:
                count   pct
time_bucket                
afternoon    15936406  47.1
morning      11388324  33.7
night         6494376  19.2


## 3. Aggregate: Product x Day x Time Bucket

In [4]:
t0 = time.time()

context_agg = (
    df.groupby(['product_name', 'department', 'aisle', 'order_dow', 'time_bucket'])
    .agg(
        purchase_count=('order_id', 'size'),
        reorder_sum=('reordered', 'sum'),
        total_orders=('reordered', 'size'),
    )
    .reset_index()
)

print(f'Aggregated to {len(context_agg):,} rows in {time.time()-t0:.1f}s')
print(f'Unique products: {context_agg["product_name"].nunique():,}')
context_agg.head()

Aggregated to 796,480 rows in 22.1s
Unique products: 49,685


,product_name,department,aisle,order_dow,time_bucket,purchase_count,reorder_sum,total_orders
0,#2 Coffee Filters,beverages,coffee,0,afternoon,68,14,68
1,#2 Coffee Filters,beverages,coffee,0,morning,53,17,53
2,#2 Coffee Filters,beverages,coffee,0,night,23,4,23
3,#2 Coffee Filters,beverages,coffee,1,afternoon,78,29,78
4,#2 Coffee Filters,beverages,coffee,1,morning,49,15,49


## 4. Compute Features

In [5]:
# 4a. Reorder Rate
context_agg['reorder_rate'] = (
    context_agg['reorder_sum'] / context_agg['total_orders']
).fillna(0)

print('Reorder rate stats:')
print(context_agg['reorder_rate'].describe().to_string())

Reorder rate stats:
count    796480.000000
mean          0.402865
std           0.311262
min           0.000000
25%           0.000000
50%           0.428571
75%           0.615385
max           1.000000


In [6]:
# 4b. Normalized Purchase Frequency (within each context)
context_agg['normalized_purchase_frequency'] = (
    context_agg.groupby(['order_dow', 'time_bucket'])['purchase_count']
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0)
)

print('Normalized purchase frequency stats:')
print(context_agg['normalized_purchase_frequency'].describe().to_string())

Normalized purchase frequency stats:
count    796480.000000
mean          0.001782
std           0.011652
min           0.000000
25%           0.000041
50%           0.000194
75%           0.000834
max           1.000000


In [7]:
# 4c. Contextual Frequency (lift vs global average)
product_global = df.groupby('product_name').size().reset_index(name='global_count')
n_contexts = df[['order_dow', 'time_bucket']].drop_duplicates().shape[0]
product_global['global_avg_per_context'] = product_global['global_count'] / n_contexts

context_agg = context_agg.merge(
    product_global[['product_name', 'global_avg_per_context']],
    on='product_name', how='left'
)

# Lift capped at 3.0
context_agg['contextual_frequency_raw'] = (
    context_agg['purchase_count'] / context_agg['global_avg_per_context']
).clip(upper=3.0)

# Normalize to 0-1
cf_min = context_agg['contextual_frequency_raw'].min()
cf_max = context_agg['contextual_frequency_raw'].max()
context_agg['contextual_frequency'] = (
    (context_agg['contextual_frequency_raw'] - cf_min) / (cf_max - cf_min)
) if cf_max > cf_min else 0

print(f'Number of contexts: {n_contexts}')
print('Contextual frequency stats:')
print(context_agg['contextual_frequency'].describe().to_string())

Number of contexts: 21
Contextual frequency stats:
count    796480.000000
mean          0.404100
std           0.243926
min           0.000000
25%           0.218109
50%           0.346507
75%           0.525633
max           1.000000


## 5. Compute Weighted Trending Score

In [8]:
context_agg['weighted_trending_score'] = (
    0.5 * context_agg['normalized_purchase_frequency']
    + 0.3 * context_agg['reorder_rate']
    + 0.2 * context_agg['contextual_frequency']
)

# Round
float_cols = ['reorder_rate', 'normalized_purchase_frequency',
              'contextual_frequency', 'weighted_trending_score']
context_agg[float_cols] = context_agg[float_cols].round(4)

# Sort
context_agg = context_agg.sort_values(
    ['order_dow', 'time_bucket', 'weighted_trending_score'],
    ascending=[True, True, False],
)

print('Weighted trending score stats:')
print(context_agg['weighted_trending_score'].describe().to_string())
print(f'\nTop 10 overall:')
context_agg.head(10)[['product_name', 'department', 'order_dow', 'time_bucket',
                       'purchase_count', 'reorder_rate', 'weighted_trending_score']]

Weighted trending score stats:
count    796480.000000
mean          0.202570
std           0.100331
min           0.000000
25%           0.132600
50%           0.200100
75%           0.268900
max           0.895200

Top 10 overall:


,product_name,department,order_dow,time_bucket,purchase_count,reorder_rate,weighted_trending_score
57228,Banana,produce,0,afternoon,50752,0.8370,0.8952
53828,Bag of Organic Bananas,produce,0,afternoon,37120,0.8279,0.7450
457165,Organic Baby Spinach,produce,0,afternoon,28701,0.7656,0.6717
512646,Organic Strawberries,produce,0,afternoon,27682,0.7753,0.6453
483391,Organic Hass Avocado,produce,0,afternoon,23129,0.7869,0.6100
456487,Organic Avocado,produce,0,afternoon,21545,0.7573,0.6028
355162,Large Lemon,produce,0,afternoon,18261,0.6938,0.5466
371567,Limes,produce,0,afternoon,16151,0.6723,0.5145
524283,Organic Yellow Onion,produce,0,afternoon,13974,0.6953,0.5121
522640,Organic Whole Milk,dairy eggs,0,afternoon,12877,0.8309,0.5016


## 6. Department-Level Trending

In [9]:
dept_agg = (
    df.groupby(['department', 'order_dow', 'time_bucket'])
    .agg(
        purchase_count=('order_id', 'size'),
        reorder_sum=('reordered', 'sum'),
        total_orders=('reordered', 'size'),
    )
    .reset_index()
)

dept_agg['reorder_rate'] = (dept_agg['reorder_sum'] / dept_agg['total_orders']).fillna(0)

dept_agg['normalized_frequency'] = (
    dept_agg.groupby(['order_dow', 'time_bucket'])['purchase_count']
    .transform(lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0)
)

dept_agg['trending_score'] = (
    0.6 * dept_agg['normalized_frequency']
    + 0.4 * dept_agg['reorder_rate']
).round(4)

dept_agg = dept_agg.sort_values(
    ['order_dow', 'time_bucket', 'trending_score'],
    ascending=[True, True, False],
)

print(f'{len(dept_agg):,} department x context rows')
dept_agg.head(10)

441 department x context rows


,department,order_dow,time_bucket,purchase_count,reorder_sum,total_orders,reorder_rate,normalized_frequency,trending_score
399,produce,0,afternoon,1023127,649277,1023127,0.634601,1.000000,0.8538
147,dairy eggs,0,afternoon,529067,345908,529067,0.653808,0.515648,0.5709
63,beverages,0,afternoon,229018,146610,229018,0.640168,0.221495,0.3890
420,snacks,0,afternoon,257166,144165,257166,0.560591,0.249090,0.3737
210,frozen,0,afternoon,238008,127385,238008,0.535213,0.230309,0.3523
42,bakery,0,afternoon,120177,74511,120177,0.620010,0.114793,0.3169
168,deli,0,afternoon,112532,67579,112532,0.600531,0.107298,0.3046
273,meat seafood,0,afternoon,80418,45606,80418,0.567112,0.075815,0.2723
21,babies,0,afternoon,39652,23298,39652,0.587562,0.035850,0.2565
84,breakfast,0,afternoon,65677,35811,65677,0.545259,0.061364,0.2549


## 7. Build Lookup Dict

In [10]:
TOP_N = 20

# Rename weighted_trending_score to weighted_score for consistency with API
scores_df = context_agg.rename(columns={'weighted_trending_score': 'weighted_score'})

lookup = {}
for (dow, tb), group in scores_df.groupby(['order_dow', 'time_bucket']):
    top = group.nlargest(TOP_N, 'weighted_score')
    products = []
    for _, row in top.iterrows():
        products.append({
            'product_name': row['product_name'],
            'department': row['department'],
            'aisle': row['aisle'],
            'score': float(row['weighted_score']),
            'purchase_count': int(row['purchase_count']),
            'reorder_rate': float(row['reorder_rate']),
        })
    lookup[(int(dow), tb)] = products

print(f'Lookup dict: {len(lookup)} contexts, {TOP_N} products each')

# Preview one context
DAY_NAMES = {0: 'Saturday', 1: 'Sunday', 2: 'Monday', 3: 'Tuesday',
             4: 'Wednesday', 5: 'Thursday', 6: 'Friday'}

sample_key = (1, 'morning')  # Sunday morning
print(f'\nSample: {DAY_NAMES[sample_key[0]]} {sample_key[1]}')
for i, p in enumerate(lookup[sample_key][:5], 1):
    print(f'  {i}. {p["product_name"]} ({p["department"]}) '
          f'score={p["score"]:.3f} reorder={p["reorder_rate"]:.1%}')

Lookup dict: 21 contexts, 20 products each

Sample: Sunday morning
  1. Banana (produce) score=0.864 reorder=86.7%
  2. Bag of Organic Bananas (produce) score=0.745 reorder=85.0%
  3. Organic Strawberries (produce) score=0.572 reorder=79.1%
  4. Organic Baby Spinach (produce) score=0.531 reorder=78.1%
  5. Organic Hass Avocado (produce) score=0.528 reorder=81.8%


## 8. Save Outputs

In [11]:
# Drop helper columns before saving
save_cols = ['product_name', 'department', 'aisle', 'order_dow', 'time_bucket',
             'purchase_count', 'reorder_rate', 'normalized_purchase_frequency',
             'contextual_frequency', 'weighted_score']
scores_save = scores_df[[c for c in save_cols if c in scores_df.columns]]

# Parquet
scores_path = os.path.join(OUTPUTS_DIR, 'trending_scores.parquet')
scores_save.to_parquet(scores_path, index=False)
print(f'Saved: {scores_path} ({os.path.getsize(scores_path)/1e6:.1f} MB)')

# CSV (top 5000 for readability)
csv_path = os.path.join(OUTPUTS_DIR, 'trending_scores.csv')
scores_save.head(5000).to_csv(csv_path, index=False)
print(f'Saved: {csv_path} (top 5000)')

# Department scores
dept_path = os.path.join(OUTPUTS_DIR, 'trending_departments.parquet')
dept_agg.to_parquet(dept_path, index=False)
print(f'Saved: {dept_path}')

# Lookup pickle
pkl_path = os.path.join(OUTPUTS_DIR, 'trending_lookup.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(lookup, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f'Saved: {pkl_path}')

print('\nAll outputs saved successfully!')

Saved: ../outputs\trending_scores.parquet (21.9 MB)
Saved: ../outputs\trending_scores.csv (top 5000)
Saved: ../outputs\trending_departments.parquet
Saved: ../outputs\trending_lookup.pkl

All outputs saved successfully!


## 9. Verification

In [12]:
# Verify outputs load correctly
loaded_scores = pd.read_parquet(scores_path)
print(f'Trending scores: {loaded_scores.shape}')
print(f'Unique products: {loaded_scores["product_name"].nunique():,}')

with open(pkl_path, 'rb') as f:
    loaded_lookup = pickle.load(f)
print(f'Lookup contexts: {len(loaded_lookup)}')

loaded_depts = pd.read_parquet(dept_path)
print(f'Department trends: {loaded_depts.shape}')

print('\nAll outputs verified!')

Trending scores: (796480, 10)
Unique products: 49,685
Lookup contexts: 21
Department trends: (441, 9)

All outputs verified!
